### Handling Catalog

In [0]:
CREATE CATALOG deltalake1;
USE CATALOG deltalake1;

In [0]:
-- Info about catalog  
DESCRIBE CATALOG EXTENDED deltalake1

info_name,info_value
Catalog Name,deltalake1
Comment,
Owner,muaazmuzammil69@gmail.com
Catalog Type,Regular
Created By,muaazmuzammil69@gmail.com
Created At,2026-04-29 AD at 14:45:08 UTC
Updated By,muaazmuzammil69@gmail.com
Updated At,2026-05-02 AD at 23:45:37 UTC
Storage Root,abfss://deltalake@muaazexternalstorage.dfs.core.windows.net/catalogs
Storage Location,abfss://deltalake@muaazexternalstorage.dfs.core.windows.net/catalogs/__unitystorage/catalogs/af2af7cb-a924-46a4-af16-08935db16feb


In [0]:
-- to drop catalog
DROP CATALOG deltalake1 CASCADE

### Handling Schema

In [0]:
CREATE SCHEMA raw;
USE schema raw;

In [0]:
-- Info about schema  
DESCRIBE SCHEMA EXTENDED raw

database_description_item,database_description_value
Catalog Name,deltalake1
Namespace Name,raw
Comment,
Location,
Owner,muaazmuzammil69@gmail.com
Properties,
Predictive Optimization,ENABLE (inherited from METASTORE metastore_azure_eastus)


In [0]:
-- Create Simple Delta Table
CREATE Table deltalake1.raw.manage_table
(
  id INT,
  order_name STRING ,
  amount INT,
  prod_id INT
)
USING DELTA

In [0]:
-- Have Created table without having any traditional database, every thinng stored in files
INSERT INTO deltalake1.raw.manage_table
VALUES (1,'order1',100,1),(2,'order2',200,2),(3,'order3',300,3)

num_affected_rows,num_inserted_rows
3,3


In [0]:
DESCRIBE EXTENDED deltalake1.raw.manage_table

col_name,data_type,comment
id,int,null
order_name,string,null
amount,int,null
prod_id,int,null
,,
# Delta Statistics Columns,,
Column Names,"id, order_name, amount, prod_id",
Column Selection Method,first-32,
,,
# Detailed Table Information,,


In [0]:
-- This creates an external Delta table stored at the specified ADLS location
CREATE Table deltalake1.raw.external_table
(
  id INT,
  order_name STRING ,
  amount INT,
  prod_id INT
)
USING DELTA
Location "abfss://deltalake@muaazexternalstorage.dfs.core.windows.net/ExternalTables/ext_table1";

INSERT INTO deltalake1.raw.external_table
VALUES (1,'order1',100,1),(2,'order2',200,2),(3,'order3',300,3)

num_affected_rows,num_inserted_rows
3,3


In [0]:
-- Query Delta table directly from the underlying storage path (ADLS)
SELECT * FROM delta.`abfss://deltalake@muaazexternalstorage.dfs.core.windows.net/ExternalTables/ext_table1`

id,order_name,amount,prod_id
1,order1,100,1
2,order2,200,2
3,order3,300,3


### CETAS , CLONE (DEEP,SHALLOW)

Need to share data with external systems (Synapse, Athena)?
→ Use CETAS

Need a full backup or migrate to new storage?
→ Use DEEP CLONE

Need a quick dev/test copy without storage cost?
→ Use SHALLOW CLONE

Need to snapshot data for ML training?
→ DEEP CLONE (stable) or SHALLOW CLONE (if source won't be vacuumed)


In [0]:
-- CETAS create table from existing one , Create a new Delta table using data from an existing table (CTAS)
-- it will create new table with same data at new/another location

-- deepcone and cetas are same thing , when we peroffrom clone it take latest versoin of that table

CREATE Table deltalake1.raw.cetas_table
USING DELTA
AS
SELECT * FROM deltalake1.raw.external_table

num_affected_rows,num_inserted_rows


In [0]:
SELECT * FROM deltalake1.raw.cetas_table

id,order_name,amount,prod_id
1,order1,100,1
2,order2,200,2
3,order3,300,3


In [0]:
-- Deep clone creates a fully independent copy of the table, including data/parquet files and Delta log files both
CREATE Table deltalake1.raw.ext_table_deep
DEEP Clone deltalake1.raw.external_table


In [0]:
SELECT * FROM deltalake1.raw.external_table

id,order_name,amount,prod_id
1,order1,100,1
2,order2,200,2
3,order3,300,3


In [0]:
-- Shallow clone is only supported for the MANAGED table type.

-- Shallow clone creates a new table using the same underlying data files (no data copy),
-- only metadata (Delta log) is duplicated and in delta log it is mentioned that take data from clone table
-- it will create table with new ID in location but with only delta log folder and when we insert/update/del data with this than only parquet files appear here 
-- if we insert data in prvious table from whch we make shallow copy than this not impact shallowed table becasue of version number

CREATE Table deltalake1.raw.manage_table_shallow
SHALLOW Clone deltalake1.raw.external_table

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8062328352952397>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', '-- shallow clone it only work on managed taable\n\n-- Shallow clone creates a new table using the same underlying data files (no data copy),\n-- only metadata (Delta log) is duplicated and in delta log it is mentioned that take data from clone table\nCREATE Table deltalake1.raw.manage_table_shallow\nSHALLOW Clone deltalake1.raw.external_table\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545

In [0]:
CREATE Table deltalake1.raw.manage_table_shallow
SHALLOW Clone deltalake1.raw.manage_table

In [0]:
SELECT * FROM deltalake1.raw.manage_table_shallow

id,order_name,amount,prod_id
1,order1,100,1
2,order2,200,2
3,order3,300,3


## DML(UPDATE , DELETE , INSERT) in Delta Tables

### Without Deletion Vector

In [0]:
-- Delta Lake DML = file level changes + transaction log tracking (ACID guarantees)

CREATE Table deltalake1.raw.external_dml_table
(
  id INT,
  order_name STRING ,
  amount INT,
  prod_id INT
)
USING DELTA
Location "abfss://deltalake@muaazexternalstorage.dfs.core.windows.net/ExternalTables/no_del_vector_dml_table"

In [0]:
-- Turninig off deletion vector
ALTER TABLE deltalake1.raw.external_dml_table SET TBLPROPERTIES ('delta.enableDeletionVectors' = False);

In [0]:
INSERT INTO deltalake1.raw.external_dml_table
VALUES (1,'order1',100,1),(2,'order2',200,2),(3,'order3',300,3)

num_affected_rows,num_inserted_rows
3,3


In [0]:
SELECT * FROM deltalake1.raw.external_dml_table

id,order_name,amount,prod_id
1,order1,100,1
2,order2,200,2
3,order3,300,3


In [0]:
-- Updating data in Delta table (deletion vectors disabled)
-- This operation rewrites the affected Parquet file:
-- a new file is created with updated records, and the old file is marked as inactive (still available for time travel)

UPDATE deltalake1.raw.external_dml_table SET amount = 1000 WHERE id = 1;

num_affected_rows
1


In [0]:
-- DELETE rewrites affected files (DV OFF); old files are tombstoned (marked inactive) and new files reflect the change
DELETE FROM deltalake1.raw.external_dml_table WHERE id = 1;

num_affected_rows
1


### DML with deletion vector ON

In [0]:
-- deletion vector add flag in records like add new column and tweak flag in same file and no new file generate or pervious removed and when perform optimize command htan just pick records and genertae new parquet file base on flags

In [0]:
 CREATE Table deltalake1.raw.delvect_external_dml_table
(
  id INT,
  order_name STRING ,
  amount INT,
  prod_id INT
)
USING DELTA
Location "abfss://deltalake@muaazexternalstorage.dfs.core.windows.net/ExternalTables/del_vector_dml_table"

In [0]:
-- Turninig ON deletion vector
ALTER TABLE deltalake1.raw.delvect_external_dml_table SET TBLPROPERTIES ('delta.enableDeletionVectors' = True);

In [0]:
INSERT INTO deltalake1.raw.delvect_external_dml_table
VALUES (1,'order1',100,1),(2,'order2',200,2),(3,'order3',300,3)

num_affected_rows,num_inserted_rows
3,3


In [0]:
UPDATE deltalake1.raw.delvect_external_dml_table SET amount = 1000 WHERE id = 1;
DELETE FROM deltalake1.raw.delvect_external_dml_table WHERE id = 1;

num_affected_rows
1


## Time Travel and Versioning

In [0]:
DESCRIBE HISTORY deltalake1.raw.delvect_external_dml_table

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-04-28T16:35:17.000Z,143405734051923,muaazmuzammil69@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2315695601999907),11f417c7-54a4-42e1-a402-03e8743675df,0428-153736-kdxbb8ql-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 2516, p25FileSize -> 1290, numDeletionVectorsRemoved -> 1, minFileSize -> 1290, numAddedFiles -> 1, maxFileSize -> 1290, p75FileSize -> 1290, p50FileSize -> 1290, numAddedBytes -> 1290)",null,Databricks-Runtime/18.1.x-photon-scala2.13
2,2026-04-28T16:35:16.000Z,143405734051923,muaazmuzammil69@gmail.com,UPDATE,"Map(predicate -> [""(id#14237 = 1)""])",null,List(2315695601999907),11f417c7-54a4-42e1-a402-03e8743675df,0428-153736-kdxbb8ql-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2167, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1152, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1250, rewriteTimeMs -> 1015)",null,Databricks-Runtime/18.1.x-photon-scala2.13
1,2026-04-28T16:34:54.000Z,143405734051923,muaazmuzammil69@gmail.com,WRITE,"Map(mode -> Append, partitionBy -> [], statsOnLoad -> false)",null,List(2315695601999907),a5b50b05-95a8-49e9-bef4-d9eeba66e2d0,0428-153736-kdxbb8ql-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputBytes -> 1266, numOutputRows -> 3)",null,Databricks-Runtime/18.1.x-photon-scala2.13
0,2026-04-28T16:34:45.000Z,143405734051923,muaazmuzammil69@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> false, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(2315695601999907),226be3c1-728a-4848-931e-7bb23ceca3bd,0428-153736-kdxbb8ql-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/18.1.x-photon-scala2.13


In [0]:
-- will just show the version 1 of table @v1
SELECT * FROM deltalake1.raw.delvect_external_dml_table@v1

In [0]:
RESTORE deltalake1.raw.delvect_external_dml_table TO VERSION AS OF 2;

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
2516,2,1,2,1290,2516


In [0]:
SELECT * FROM deltalake1.raw.delvect_external_dml_table

id,order_name,amount,prod_id
2,order2,200,2
3,order3,300,3
1,order1,1000,1


In [0]:
DESCRIBE EXTENDED deltalake1.raw.delvect_external_dml_table

col_name,data_type,comment
id,int,null
order_name,string,null
amount,int,null
prod_id,int,null
,,
# Delta Statistics Columns,,
Column Names,"id, order_name, amount, prod_id",
Column Selection Method,first-32,
,,
# Detailed Table Information,,


### Vacuum

In [0]:
-- Dry run: lists files that would be removed by VACUUM without actually deleting them
VACUUM deltalake1.raw.delvect_external_dml_table RETAIN 0 HOURS DRY RUN;

-- Actual vacuum: permanently deletes files no longer referenced by Delta table
VACUUM deltalake1.raw.delvect_external_dml_table RETAIN 0 HOURS;


In [0]:
-- Disable Delta safety check to allow aggressive VACUUM (retention = 0 hours)
-- Normally, Delta Lake enforces a minimum retention period (default is 7 days) to protect
SET spark.databricks.delta.retentionDurationCheck.enabled = false;

### Optimize Command

In [0]:
 CREATE Table deltalake1.raw.optimize_dml_table
(
  id INT,
  order_name STRING ,
  amount INT,
  prod_id INT
)
USING DELTA
Location "abfss://deltalake@muaazexternalstorage.dfs.core.windows.net/ExternalTables/optimize_dml_table"

In [0]:
INSERT INTO deltalake1.raw.optimize_dml_table
VALUES (1,'order1',100,1),(2,'order2',200,2),(3,'order3',300,3)

num_affected_rows,num_inserted_rows
3,3


In [0]:
INSERT INTO deltalake1.raw.optimize_dml_table
VALUES (4,'order1',100,1),(5,'order2',200,2),(6,'order3',300,3)

num_affected_rows,num_inserted_rows
3,3


In [0]:
INSERT INTO deltalake1.raw.optimize_dml_table
VALUES (7,'order1',100,1),(8,'order2',200,2),(9,'order3',300,3)

num_affected_rows,num_inserted_rows
3,3


In [0]:
Optimize deltalake1.raw.optimize_dml_table

path,metrics
abfss://iot@muaazexternalstorage.dfs.core.windows.net/ExternalTables/optimize_dml_table,"List(1, 3, List(1414, 1414, 1414.0, 1, 1414), List(1266, 1266, 1266.0, 3, 3798), 0, null, null, 0, 1, 3, 0, true, 0, 0, 1777476710364, 1777476712641, 8, 1, null, List(0, 0), null, 4, 4, 745, 0, null, null)"


In [0]:
-- Zorder BY
Optimize deltalake1.raw.optimize_dml_table zorder by id

### Liquid Clustering

In [0]:
-- 

In [0]:
 CREATE Table deltalake1.raw.liquid_table
(
  id INT,
  order_name STRING ,
  amount INT,
  prod_id INT
) 
USING DELTA
Location "abfss://deltalake@muaazexternalstorage.dfs.core.windows.net/ExternalTables/liquid_table"
CLUSTER BY (id)

## Schema Enforcement & Evolution

In [0]:
%python
my_data=[(1,'food',10),(2, 'food', 20),(3,'food',30),(4,'food',40),(5,'food',50),(6,'food',60),(7,'food',70),(8,'food',80)]
my_schema="id int, category string, value int"
my_df = spark.createDataFrame(my_data, my_schema)
display(my_df)

id,category,value
1,food,10
2,food,20
3,food,30
4,food,40
5,food,50
6,food,60
7,food,70
8,food,80


In [0]:
%python
my_df.write.format("delta").mode("append").option("path","abfss://deltalake@muaazexternalstorage.dfs.core.windows.net/ExternalTables/my_df").save()


In [0]:
%python
# Insert data in dataframe , will easily add new_datframe in that delta table because schema is same
new_df=my_df.union(spark.createDataFrame([(9,'food',90),(10,'food',100)],my_schema))
new_df.write.format("delta").mode("append").option("path","abfss://deltalake@muaazexternalstorage.dfs.core.windows.net/ExternalTables/my_df").save()

In [0]:
%python
# will thorugh an error as the schema is not same as the existing table 
from pyspark.sql.functions import lit
df_new=new_df.withColumn("flag",lit(1))
df_new.write.format("delta").mode("append").save("abfss://deltalake@muaazexternalstorage.dfs.core.windows.net/ExternalTables/my_df")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7093790027664178>, line 4
      2 from pyspark.sql.functions import lit
      3 df_new=new_df.withColumn("flag",lit(1))
----> 4 df_new.write.format("delta").mode("append").save("abfss://iot@muaazexternalstorage.dfs.core.windows.net/deltalake/my_df")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:703, in DataFrameWriter.save(self, path, format, mode, partitionBy, **options)
    701     self.format(format)
    702 self._write.path = path
--> 703 _, _, ei = self._spark.client.execute_command(
    704     self._write.command(self._spark.client), self._write.observations
    705 )
    706 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadat

In [0]:
%python
# Merge schema , now ownward delta log consider this schema not previous one
from pyspark.sql.functions import lit
df_new=new_df.withColumn("flag",lit(1))
df_new.write.format("delta").mode("append").option("mergeSchema","true").save("abfss://deltalake@muaazexternalstorage.dfs.core.windows.net/ExternalTables/my_df")

In [0]:
%python
df=spark.read.format("delta").load("abfss://deltalake@muaazexternalstorage.dfs.core.windows.net/ExternalTables/my_df")
display(df)

id,category,value,flag
1,food,10,1
2,food,20,1
3,food,30,1
4,food,40,1
5,food,50,1
6,food,60,1
7,food,70,1
8,food,80,1
9,food,90,1
10,food,100,1


## Structured Streaming in Delta Tables
## 

In [0]:
 CREATE Table deltalake1.raw.stream_source
(
  id INT,
  order_name STRING ,
  amount INT,
  prod_id INT
) 
USING DELTA
Location "abfss://deltalake@muaazexternalstorage.dfs.core.windows.net/ExternalTables/stream_source"

In [0]:
 INSERT INTO deltalake1.raw.stream_source
VALUES (1,'order1',100,1),(2,'order2',200,2),(3,'order3',300,3)

num_affected_rows,num_inserted_rows
3,3


In [0]:
%python
df=spark.readStream.table("deltalake1.raw.stream_source")

df.writeStream
  .format("delta")
  .option("checkpointLocation","abfss://deltalake@muaazexternalstorage.dfs.core.windows.net/ExternalTables/streamsink/checkpoint")
  .trigger(processingTime='10 seconds')
  .start("abfss://deltalake@muaazexternalstorage.dfs.core.windows.net/ExternalTables/streamsink/data")

In [0]:
SELECT * FROM delta.`abfss://deltalake@muaazexternalstorage.dfs.core.windows.net/ExternalTables/streamsink/data`

id,order_name,amount,prod_id
1,order1,100,1
2,order2,200,2
3,order3,300,3


## Autoloader

In [0]:
%python
from pyspark.sql.functions import *

In [0]:
%python
df=spark.readStream.format("Cloudfiles")\
  .option("cloudFiles.format","parquet")\
  .option("cloudFiles.schemaLocation","abfss://deltalake@muaazexternalstorage.dfs.core.windows.net/autoloadersink/check")\
  .load("abfss://deltalake@muaazexternalstorage.dfs.core.windows.net/autoloadersource")

In [0]:
%python
df.writeStream\
    .format("parquet")\
    .option("checkpointLocation", "abfss://deltalake@muaazexternalstorage.dfs.core.windows.net/autoloadersink/check")\
    .trigger(processingTime='10 seconds')\
    .start("abfss://deltalake@muaazexternalstorage.dfs.core.windows.net/autoloadersink/data")

## Python APIs for Delta Lake

In [0]:
%python
# above i was using SQL to create worrk with delta this is for python API of delta lake

deltaTable = DeltaTable.forPath(spark, "/path/to/table")


## Register Existing External Tables in the Metastore

In [0]:
%python
# Register table in metastore from external location (table are present in storage will regidter in catalog)

df = spark.read.format("delta").load("abfss://deltalake@muaazexternalstorage.dfs.core.windows.net/tables/41094116-d61a-4e1f-9865-cdd9d69f6341")

df.writeTo("externallcatalog.schematest.customer") \
.using("delta") \
.createOrReplace()

## CDF (Change Data Feed)

In [0]:
%sql
CREATE Table cdfcatalog.schema1.cdf_table
(
  id INT,
  order_name STRING ,
  amount INT,
  prod_id INT
)
USING DELTA;

INSERT INTO cdfcatalog.schema1.cdf_table
VALUES (1,'order1',100,1),(2,'order2',200,2),(3,'order3',300,3);

SELECT * FROM cdfcatalog.schema1.cdf_table

In [0]:
ALTER TABLE cdfcatalog.schema1.cdf_table SET TBLPROPERTIES (delta.enableChangeDataFeed = true)

In [0]:
INSERT INTO cdfcatalog.schema1.cdf_table
VALUES (5,'order5',500,1),(6,'order6',600,2),(7,'order7',700,3);

UPDATE cdfcatalog.schema1.cdf_table SET amount=1000 WHERE id=3;
UPDATE cdfcatalog.schema1.cdf_table SET amount=1200 WHERE id=1;

DELETE FROM cdfcatalog.schema1.cdf_table WHERE id=5

In [0]:
starting_version=2
SELECT * FROM table_changes('cdfcatalog.schema1.cdf_table', starting_version)

## MERGE / UPSERT

In [0]:

-- helpful in Incremental data loading
-- MERGE / UPSERT
-- -> target table 
-- -> source table

USE CATALOG deltalake1;

USE SCHEMA raw;

CREATE table destination
(
  id int,
  category string,
  value int
);
INSERT INTO destination
VALUES
(1,'A',10),
(2,'B',20),
(3,'C',30),
(4,'D',40);

num_affected_rows,num_inserted_rows
4,4


In [0]:
CREATE table source
(
  id int,
  category string,
  value int
);
INSERT INTO source
VALUES
(1,'A',100),
(2,'B',200),
(3,'C',300),
(5,'F',500);

num_affected_rows,num_inserted_rows
4,4


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4,3,0,1


In [0]:
-- when we need to delete records from target that are not in source
MERGE INTO destination d
USING source s
ON d.id = s.id
WHEN MATCHED THEN
UPDATE SET d.category = s.category,d.value = s.value
WHEN NOT MATCHED THEN
INSERT *
WHEN NOT MATCHED by destination THEN
DELETE

-- we can also use MERGE INTO to perform DELETE operation

id,category,value
4,D,40
1,A,100
2,B,200
3,C,300
5,F,500
